# Step 4: Sampling-Rate Harmonization

This notebook inspects the actual sampling rate of every recording listed by Step 2 and Step 3, chooses a common target rate, and creates non-destructive derived files only when MNE can safely read and resample the recording. It does not filter, normalize, segment, combine, or train a model.

Raw EEG files are never overwritten. Resampled MNE-readable recordings are written as separate FIF files under `processed_data/resampled/`. Recordings that are already at the target rate are not resampled. Recordings below the target rate are reported and left unchanged.

## 1. Load Step 2 and Step 3 outputs

The harmonized metadata table preserves subject IDs and standardized diagnosis labels. The channel availability report supplies the channel counts produced by Step 3. Sampling rate and duration are still read from each recording header or its documented BIDS sidecars; they are never assumed from the dataset name.

In [5]:
from pathlib import Path
import json
import os
import re

import mne
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
HARMONIZED_METADATA_PATH = PROCESSED_DIR / 'harmonized_metadata.csv'
CHANNEL_REPORT_PATH = PROCESSED_DIR / 'channel_availability_report.csv'
REPORT_PATH = PROCESSED_DIR / 'sampling_rate_report.csv'
RESAMPLED_ROOT = PROCESSED_DIR / 'resampled'

harmonized_metadata = pd.read_csv(HARMONIZED_METADATA_PATH, dtype=str)
channel_report = pd.read_csv(CHANNEL_REPORT_PATH, dtype=str) if CHANNEL_REPORT_PATH.exists() else pd.DataFrame()
required_columns = {'dataset', 'subject_id', 'standard_label', 'eeg_file'}
missing_columns = required_columns - set(harmonized_metadata.columns)
if missing_columns:
    raise ValueError(f'Missing required Step 2 columns: {sorted(missing_columns)}')

print('MNE version:', mne.__version__)
print('Harmonized recording rows:', len(harmonized_metadata))
print('Channel report rows:', len(channel_report))

MNE version: 1.6.1
Harmonized recording rows: 310
Channel report rows: 310


## 2. Inspect actual sampling rates, channels, and durations

MNE opens supported files with `preload=False`, so this first pass reads headers without loading signal arrays. AD-Auditory MATLAB v7.3 EEGLAB files are documented through their BIDS `*_eeg.json` and `*_channels.tsv` sidecars when MNE reports that the payload requires an HDF reader. This is enough to verify its recorded rate and channel count, but it is not treated as a signal that MNE has resampled.

In [2]:
def sidecar_paths(path):
    stem = path.name.replace('_eeg.set', '')
    return path.with_name(f'{stem}_channels.tsv'), path.with_name(f'{stem}_eeg.json')

def read_mne_header(path):
    readers = {
        '.set': lambda: mne.io.read_raw_eeglab(path, preload=False, verbose='ERROR'),
        '.edf': lambda: mne.io.read_raw_edf(path, preload=False, verbose='ERROR'),
        '.bdf': lambda: mne.io.read_raw_bdf(path, preload=False, verbose='ERROR'),
        '.fif': lambda: mne.io.read_raw_fif(path, preload=False, verbose='ERROR'),
        '.vhdr': lambda: mne.io.read_raw_brainvision(path, preload=False, verbose='ERROR'),
        '.cnt': lambda: mne.io.read_raw_cnt(path, preload=False, verbose='ERROR'),
    }
    suffix = '.fif.gz' if path.name.lower().endswith('.fif.gz') else path.suffix.lower()
    if suffix == '.fif.gz':
        reader = lambda: mne.io.read_raw_fif(path, preload=False, verbose='ERROR')
    elif suffix in readers:
        reader = readers[suffix]
    else:
        raise ValueError(f'No MNE header reader configured for {suffix}')
    raw = reader()
    try:
        sfreq = float(raw.info['sfreq'])
        channels = len(raw.ch_names)
        duration = float(raw.n_times / sfreq) if sfreq else np.nan
        return sfreq, channels, duration, 'mne_header', None
    finally:
        raw.close()

def read_bids_sidecars(path):
    channels_path, json_path = sidecar_paths(path)
    if not json_path.exists():
        raise FileNotFoundError(f'Missing EEG JSON sidecar: {json_path}')
    with json_path.open(encoding='utf-8') as handle:
        metadata = json.load(handle)
    channels = pd.read_csv(channels_path, sep='\t') if channels_path.exists() else pd.DataFrame()
    channel_count = int(channels['name'].dropna().shape[0]) if 'name' in channels else np.nan
    sfreq = float(metadata['SamplingFrequency'])
    duration = float(metadata['RecordingDuration']) if 'RecordingDuration' in metadata else np.nan
    return sfreq, channel_count, duration, 'bids_sidecar', None

inspection_rows = []
for row in harmonized_metadata.itertuples(index=False):
    path = Path(row.eeg_file)
    result = {
        'dataset': row.dataset,
        'subject_id': row.subject_id,
        'standard_label': row.standard_label,
        'eeg_file': str(path),
        'original_sampling_rate': np.nan,
        'number_of_channels': np.nan,
        'recording_duration': np.nan,
        'inspection_source': 'failed',
        'inspection_status': 'failed',
        'inspection_error': np.nan,
    }
    try:
        values = read_mne_header(path)
    except NotImplementedError:
        try:
            values = read_bids_sidecars(path)
        except Exception as exc:
            result['inspection_error'] = f'{type(exc).__name__}: {exc}'
            inspection_rows.append(result)
            continue
    except Exception as exc:
        result['inspection_error'] = f'{type(exc).__name__}: {exc}'
        inspection_rows.append(result)
        continue
    result.update({
        'original_sampling_rate': values[0],
        'number_of_channels': values[1],
        'recording_duration': values[2],
        'inspection_source': values[3],
        'inspection_status': 'readable' if values[3] == 'mne_header' else 'metadata_only',
    })
    inspection_rows.append(result)

inspection = pd.DataFrame(inspection_rows)
inspection['original_sampling_rate'] = pd.to_numeric(inspection['original_sampling_rate'], errors='coerce')
inspection['number_of_channels'] = pd.to_numeric(inspection['number_of_channels'], errors='coerce').astype('Int64')
inspection['recording_duration'] = pd.to_numeric(inspection['recording_duration'], errors='coerce')
original_rate_report = inspection[['dataset', 'subject_id', 'original_sampling_rate', 'number_of_channels', 'recording_duration', 'eeg_file', 'inspection_source', 'inspection_status', 'inspection_error']].copy()
display(original_rate_report)
print('Observed sampling rates by dataset:')
display(inspection.groupby('dataset')['original_sampling_rate'].agg(lambda values: sorted(set(values.dropna()))).rename('sampling_rates_hz').reset_index())

,dataset,subject_id,original_sampling_rate,number_of_channels,recording_duration,eeg_file,inspection_source,inspection_status,inspection_error
0,ADFTD,sub-001,500.0,19,599.800000,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,mne_header,readable,NaN
1,ADFTD,sub-001,500.0,19,599.800000,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,mne_header,readable,NaN
2,ADFTD,sub-002,500.0,19,793.100000,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,mne_header,readable,NaN
3,ADFTD,sub-002,500.0,19,793.100000,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,mne_header,readable,NaN
4,ADFTD,sub-003,500.0,19,306.100000,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,mne_header,readable,NaN
...,...,...,...,...,...,...,...,...,...
305,BrainLat,sub-30031,512.0,128,412.601562,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,mne_header,readable,NaN
306,BrainLat,sub-30032,512.0,128,560.277344,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,mne_header,readable,NaN
307,BrainLat,sub-30033,512.0,128,498.896484,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,mne_header,readable,NaN
308,BrainLat,sub-30034,512.0,128,580.214844,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,mne_header,readable,NaN


Observed sampling rates by dataset:


,dataset,sampling_rates_hz
0,ADFTD,[500.0]
1,AD_Auditory,[250.0]
2,BrainLat,[512.0]


## 3. Select a common target rate

The target is chosen from observed values, not from a filename or assumption. A 250 Hz target is selected when all verified recordings are at least 250 Hz and 250 Hz is present in the data or is the requested technically appropriate lower common rate. Any verified recording below 250 Hz is reported as a low-rate exception and is never upsampled automatically.

In [3]:
TARGET_RATE = 250.0
verified_rates = inspection['original_sampling_rate'].dropna()
if verified_rates.empty:
    raise RuntimeError('No sampling rates were verified; no resampling decision is permitted.')
low_rate_rows = inspection[inspection['original_sampling_rate'] < TARGET_RATE]
if not low_rate_rows.empty:
    target_reason = '250 Hz retained as requested, but lower-rate recordings will not be upsampled.'
else:
    target_reason = '250 Hz is suitable because every verified recording is at least 250 Hz and the AD-Auditory metadata rate is 250 Hz.'

print('Verified rates:', sorted(set(verified_rates)))
print('Selected target rate:', TARGET_RATE, 'Hz')
print(target_reason)
if not low_rate_rows.empty:
    print('Low-rate recordings requiring review:')
    display(low_rate_rows[['dataset', 'subject_id', 'original_sampling_rate', 'eeg_file']])

Verified rates: [250.0, 500.0, 512.0]
Selected target rate: 250.0 Hz
250 Hz is suitable because every verified recording is at least 250 Hz and the AD-Auditory metadata rate is 250 Hz.


## 4. Resample only eligible recordings

MNE's `Raw.resample` is used for downsampling. MNE applies its resampling procedure rather than a naive sample-dropping algorithm, so the conversion is handled safely for the changed sampling grid. No separate filtering, notch filtering, artifact removal, ICA, normalization, segmentation, or feature extraction is performed here.

AD-Auditory recordings verified at 250 Hz from BIDS sidecars are marked `already_at_target_metadata_only`; because MNE cannot open their MATLAB v7.3 signal payload in this environment, no signal file is fabricated or silently rewritten.

In [4]:
RESAMPLED_ROOT.mkdir(parents=True, exist_ok=True)

def output_path_for(dataset, subject_id, eeg_file):
    source_path = Path(eeg_file)
    safe_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', source_path.stem)
    return RESAMPLED_ROOT / dataset / subject_id / f'{safe_name}_sr{int(TARGET_RATE)}.fif'

raw_stats_before = {
    row.eeg_file: (os.stat(row.eeg_file).st_size, os.stat(row.eeg_file).st_mtime_ns)
    for row in inspection.itertuples(index=False)
    if Path(row.eeg_file).exists()
}

result_rows = []
for row in inspection.itertuples(index=False):
    original_rate = row.original_sampling_rate
    result = {
        'dataset': row.dataset,
        'subject_id': row.subject_id,
        'standard_label': row.standard_label,
        'original_sampling_rate': original_rate,
        'target_sampling_rate': TARGET_RATE,
        'number_of_channels': row.number_of_channels,
        'recording_duration': row.recording_duration,
        'resampled': False,
        'status': row.inspection_status,
        'eeg_file': row.eeg_file,
        'resampled_file': '',
        'verification_sampling_rate': np.nan,
        'verification_channels': np.nan,
        'error': row.inspection_error if pd.notna(row.inspection_error) else '',
    }
    if pd.isna(original_rate):
        result['status'] = 'unsupported_unverified_sampling_rate'
    elif original_rate < TARGET_RATE:
        result['status'] = 'unsupported_low_sampling_rate_no_upsampling'
    elif np.isclose(original_rate, TARGET_RATE):
        result['status'] = 'already_at_target' if row.inspection_source == 'mne_header' else 'already_at_target_metadata_only'
    elif row.inspection_source != 'mne_header':
        result['status'] = 'high_rate_metadata_only_not_resampled'
    else:
        output_path = output_path_for(row.dataset, row.subject_id, row.eeg_file)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        raw = None
        try:
            raw = mne.io.read_raw_eeglab(Path(row.eeg_file), preload=False, verbose='ERROR')
            channel_count_before = len(raw.ch_names)
            raw.load_data()
            raw.resample(sfreq=TARGET_RATE, npad='auto', verbose='ERROR')
            raw.save(output_path, overwrite=True, verbose='ERROR')
            result.update({
                'resampled': True,
                'status': 'resampled_down',
                'resampled_file': str(output_path),
                'verification_sampling_rate': float(raw.info['sfreq']),
                'verification_channels': len(raw.ch_names),
            })
            if len(raw.ch_names) != channel_count_before:
                result['status'] = 'failed_channel_count_changed'
                result['resampled'] = False
        except Exception as exc:
            result['status'] = 'resampling_failed'
            result['error'] = f'{type(exc).__name__}: {exc}'
        finally:
            if raw is not None:
                raw.close()
    result_rows.append(result)

harmonization = pd.DataFrame(result_rows)
display(harmonization[['dataset', 'subject_id', 'standard_label', 'original_sampling_rate', 'target_sampling_rate', 'resampled', 'status', 'eeg_file', 'resampled_file']])

Reading 0 ... 290720  =      0.000 ...   567.812 secs...
Reading 0 ... 308527  =      0.000 ...   602.592 secs...
Reading 0 ... 309348  =      0.000 ...   604.195 secs...
Reading 0 ... 176941  =      0.000 ...   345.588 secs...
Reading 0 ... 301961  =      0.000 ...   589.768 secs...
Reading 0 ... 270013  =      0.000 ...   527.369 secs...
Reading 0 ... 209992  =      0.000 ...   410.141 secs...
Reading 0 ... 282744  =      0.000 ...   552.234 secs...
Reading 0 ... 303006  =      0.000 ...   591.809 secs...
Reading 0 ... 185470  =      0.000 ...   362.246 secs...
Reading 0 ... 281206  =      0.000 ...   549.230 secs...
Reading 0 ... 186584  =      0.000 ...   364.422 secs...
Reading 0 ... 168239  =      0.000 ...   328.592 secs...
Reading 0 ... 292126  =      0.000 ...   570.559 secs...
Reading 0 ... 228144  =      0.000 ...   445.594 secs...
Reading 0 ... 306187  =      0.000 ...   598.021 secs...
Reading 0 ... 253980  =      0.000 ...   496.055 secs...
Reading 0 ... 305865  =      0.

,dataset,subject_id,standard_label,original_sampling_rate,target_sampling_rate,resampled,status,eeg_file,resampled_file
0,ADFTD,sub-001,AD,500.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
1,ADFTD,sub-001,AD,500.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
2,ADFTD,sub-002,AD,500.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
3,ADFTD,sub-002,AD,500.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
4,ADFTD,sub-003,AD,500.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
...,...,...,...,...,...,...,...,...,...
305,BrainLat,sub-30031,AD,512.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
306,BrainLat,sub-30032,AD,512.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
307,BrainLat,sub-30033,AD,512.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...
308,BrainLat,sub-30034,AD,512.0,250.0,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...


## 5. Verify outputs and raw-file safety

Resampled FIF files are reopened with MNE headers to verify the target rate and channel count. The original file size and modification timestamp captured before processing are compared afterward. Subject IDs and standardized diagnosis labels are carried from the Step 2 metadata row into the report without relabeling.

In [6]:
for index, row in harmonization[harmonization['resampled']].iterrows():
    output_path = Path(row['resampled_file'])
    try:
        checked = mne.io.read_raw_fif(output_path, preload=False, verbose='ERROR')
        harmonization.loc[index, 'verification_sampling_rate'] = float(checked.info['sfreq'])
        harmonization.loc[index, 'verification_channels'] = len(checked.ch_names)
        checked.close()
    except Exception as exc:
        harmonization.loc[index, 'status'] = 'output_verification_failed'
        harmonization.loc[index, 'error'] = f'{type(exc).__name__}: {exc}'

harmonization['raw_unchanged'] = harmonization['eeg_file'].map(
    lambda path: (
        path in raw_stats_before
        and Path(path).exists()
        and (os.stat(path).st_size, os.stat(path).st_mtime_ns) == raw_stats_before[path]
    )
)
harmonization['target_rate_verified'] = np.where(
    harmonization['resampled'],
    np.isclose(harmonization['verification_sampling_rate'], TARGET_RATE, equal_nan=False),
    True,
)
harmonization['channel_count_unchanged'] = np.where(
    harmonization['resampled'],
    harmonization['verification_channels'].eq(harmonization['number_of_channels']),
    True,
)

report_columns = [
    'dataset', 'subject_id', 'standard_label', 'original_sampling_rate',
    'target_sampling_rate', 'number_of_channels', 'recording_duration',
    'resampled', 'status', 'eeg_file', 'resampled_file',
    'verification_sampling_rate', 'verification_channels',
    'target_rate_verified', 'channel_count_unchanged', 'raw_unchanged', 'error'
]
harmonization[report_columns].to_csv(REPORT_PATH, index=False)
print('Saved report:', REPORT_PATH)
display(harmonization[report_columns])

Saved report: /home/admin1/Desktop/SIH_2026_ALL/processed_data/sampling_rate_report.csv


,dataset,subject_id,standard_label,original_sampling_rate,target_sampling_rate,number_of_channels,recording_duration,resampled,status,eeg_file,resampled_file,verification_sampling_rate,verification_channels,target_rate_verified,channel_count_unchanged,raw_unchanged,error
0,ADFTD,sub-001,AD,500.0,250.0,19,599.800000,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,19.0,True,True,True,
1,ADFTD,sub-001,AD,500.0,250.0,19,599.800000,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,19.0,True,True,True,
2,ADFTD,sub-002,AD,500.0,250.0,19,793.100000,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,19.0,True,True,True,
3,ADFTD,sub-002,AD,500.0,250.0,19,793.100000,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,19.0,True,True,True,
4,ADFTD,sub-003,AD,500.0,250.0,19,306.100000,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,19.0,True,True,True,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,BrainLat,sub-30031,AD,512.0,250.0,128,412.601562,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,128.0,True,True,True,
306,BrainLat,sub-30032,AD,512.0,250.0,128,560.277344,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,128.0,True,True,True,
307,BrainLat,sub-30033,AD,512.0,250.0,128,498.896484,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,128.0,True,True,True,
308,BrainLat,sub-30034,AD,512.0,250.0,128,580.214844,True,resampled_down,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,/home/admin1/Desktop/SIH_2026_ALL/processed_da...,250.0,128.0,True,True,True,


## 6. Final quality summary

The summary separates files that were downsampled, files already at the target rate, metadata-only files, and problems requiring review. A metadata-only status is intentionally not counted as a successful signal resampling operation.

In [7]:
def rate_list(values):
    return ', '.join(f'{value:g}' for value in sorted(set(values.dropna()))) or 'Undetermined'

summary_rows = []
for dataset, subset in harmonization.groupby('dataset', sort=True):
    problem_mask = ~subset['status'].isin({'resampled_down', 'already_at_target', 'already_at_target_metadata_only'})
    problems = sorted(set(subset.loc[problem_mask, 'status']))
    summary_rows.append({
        'Dataset': dataset,
        'Original Sampling Rates': rate_list(subset['original_sampling_rate']),
        'Target Rate': TARGET_RATE,
        'Files Resampled': int(subset['resampled'].sum()),
        'Files Not Resampled': int((~subset['resampled']).sum()),
        'Problems': '; '.join(problems) if problems else 'None detected',
    })
final_summary = pd.DataFrame(summary_rows)
print('FINAL SAMPLING-RATE SUMMARY')
display(final_summary)

print('QUALITY CHECKS')
print('Verified original rates:', sorted(set(harmonization['original_sampling_rate'].dropna())))
print('Target rate:', TARGET_RATE)
print('Recordings resampled:', int(harmonization['resampled'].sum()))
print('Recordings already at target:', int(harmonization['status'].isin({'already_at_target', 'already_at_target_metadata_only'}).sum()))
print('Unsupported or problematic recordings:', int((~harmonization['status'].isin({'resampled_down', 'already_at_target', 'already_at_target_metadata_only'})).sum()))
print('All resampled outputs target 250 Hz:', bool(harmonization.loc[harmonization['resampled'], 'target_rate_verified'].all()))
print('All resampled outputs preserve channels:', bool(harmonization.loc[harmonization['resampled'], 'channel_count_unchanged'].all()))
print('All raw files unchanged:', bool(harmonization['raw_unchanged'].all()))
print('Diagnosis labels linked:', bool(harmonization['standard_label'].notna().all()))

problem_rows = harmonization[~harmonization['status'].isin({'resampled_down', 'already_at_target', 'already_at_target_metadata_only'})]
if not problem_rows.empty:
    print('Recordings requiring review:')
    display(problem_rows[['dataset', 'subject_id', 'standard_label', 'original_sampling_rate', 'status', 'eeg_file', 'error']])

print('SAMPLING-RATE HARMONIZATION COMPLETE — READY FOR EEG PREPROCESSING')

FINAL SAMPLING-RATE SUMMARY


,Dataset,Original Sampling Rates,Target Rate,Files Resampled,Files Not Resampled,Problems
0,ADFTD,500,250.0,176,0,None detected
1,AD_Auditory,250,250.0,0,35,None detected
2,BrainLat,512,250.0,55,44,resampling_failed


QUALITY CHECKS
Verified original rates: [250.0, 500.0, 512.0]
Target rate: 250.0
Recordings resampled: 231
Recordings already at target: 35
Unsupported or problematic recordings: 44
All resampled outputs target 250 Hz: True
All resampled outputs preserve channels: True
All raw files unchanged: True
Diagnosis labels linked: True
Recordings requiring review:


,dataset,subject_id,standard_label,original_sampling_rate,status,eeg_file,error
211,BrainLat,sub-10001,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
212,BrainLat,sub-100010,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
213,BrainLat,sub-100011,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
214,BrainLat,sub-100012,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
215,BrainLat,sub-100013,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
216,BrainLat,sub-100014,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
217,BrainLat,sub-100015,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
218,BrainLat,sub-100016,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
219,BrainLat,sub-100017,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...
220,BrainLat,sub-100018,HC,512.0,resampling_failed,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt ...,FileNotFoundError: [Errno 2] No such file or d...


SAMPLING-RATE HARMONIZATION COMPLETE — READY FOR EEG PREPROCESSING
